# 01 — Extraction Demo

Pipeline: **PDF → detect type → extract text blocks → filter noise**

This notebook demonstrates Layer 1 of the HSC-Edu pipeline on the textbook PDFs in `data/`.

In [1]:
from pathlib import Path

from hsc_edu.core.extraction import detect_pdf_type, extract_document

DATA_DIR = Path("../data")
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(PDF_FILES)} PDF(s):")
for p in PDF_FILES:
    print(f"  • {p.name}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")

Found 3 PDF(s):
  • C.pdf  (2.3 MB)
  • Java.pdf  (6.3 MB)
  • Python.pdf  (36.4 MB)


## 1. PDF Type Detection

Run `detect_pdf_type` on every PDF to check whether they are text-based, scanned, or mixed.

In [2]:
for pdf in PDF_FILES:
    r = detect_pdf_type(pdf)
    print(
        f"{pdf.name:<20s}  type={r.pdf_type.value:<12s}  "
        f"pages={r.total_pages}  text_pages={r.text_pages}  "
        f"scan_pages={r.scan_pages}  ratio={r.text_ratio:.2f}"
    )

C.pdf                 type=text-based    pages=102  text_pages=97  scan_pages=5  ratio=0.95
Java.pdf              type=text-based    pages=241  text_pages=231  scan_pages=10  ratio=0.96
Python.pdf            type=scanned       pages=219  text_pages=0  scan_pages=219  ratio=0.00


## 2. Text Block Extraction

Extract blocks from one PDF and inspect the results. Change `SAMPLE_PDF` to try a different file.

In [3]:
# Python.pdf is scanned (image-only) → use Java.pdf or C.pdf for text extraction demo
SAMPLE_PDF = DATA_DIR / "Java.pdf"

blocks = extract_document(SAMPLE_PDF, doc_id="java-demo")
print(f"Total blocks extracted from '{SAMPLE_PDF.name}': {len(blocks)}")

Total blocks extracted from 'Java.pdf': 2121


### 2.1 Statistics

In [4]:
from collections import Counter

pages = [b.page for b in blocks]
page_counts = Counter(pages)

num_pages = max(pages) + 1 if pages else 0
avg_blocks = len(blocks) / num_pages if num_pages else 0

font_sizes = [b.font_info.size for b in blocks if b.font_info]
bold_count = sum(1 for b in blocks if b.font_info and b.font_info.is_bold)

print(f"Pages covered        : {num_pages}")
print(f"Blocks per page (avg): {avg_blocks:.1f}")
print(f"Bold blocks          : {bold_count} / {len(blocks)}")
print(f"Font sizes found     : {sorted(set(round(s, 1) for s in font_sizes))}")

Pages covered        : 241
Blocks per page (avg): 8.8
Bold blocks          : 0 / 2121
Font sizes found     : [6.7, 6.8, 7.0, 7.1, 7.2, 7.3, 7.4, 7.6, 7.7, 7.8, 7.9, 8.0, 8.2, 8.3, 8.5, 8.8, 8.9, 9.0, 9.1, 9.2, 9.4, 9.5, 9.6, 9.8, 10.1, 10.4, 10.6, 10.7, 11.3, 11.5, 12.2, 15.0]


### 2.2 Blocks per page distribution

In [5]:
top_10 = page_counts.most_common(10)
print("Top 10 pages by block count:")
for pg, cnt in top_10:
    bar = "█" * cnt
    print(f"  Page {pg:>3d}: {cnt:>3d} blocks  {bar}")

Top 10 pages by block count:
  Page   0:  31 blocks  ███████████████████████████████
  Page   1:  31 blocks  ███████████████████████████████
  Page   2:  31 blocks  ███████████████████████████████
  Page 238:  29 blocks  █████████████████████████████
  Page   3:  28 blocks  ████████████████████████████
  Page  81:  26 blocks  ██████████████████████████
  Page  17:  25 blocks  █████████████████████████
  Page  29:  24 blocks  ████████████████████████
  Page  44:  24 blocks  ████████████████████████
  Page  73:  24 blocks  ████████████████████████


## 3. Sample Blocks by Page

Print the first 10 blocks to inspect `raw_text`, `bbox`, and `font_info`.

In [6]:
SHOW_N = 10

for i, b in enumerate(blocks[:SHOW_N]):
    fi = b.font_info
    font_desc = f"{fi.name}, {fi.size}pt, bold={fi.is_bold}" if fi else "N/A"
    text_preview = b.raw_text[:120].replace("\n", " ↵ ")
    if len(b.raw_text) > 120:
        text_preview += " …"

    print(f"─── Block {i} ── page {b.page} ── bbox {b.bbox} ──")
    print(f"  Font : {font_desc}")
    print(f"  Text : {text_preview}")
    print()

─── Block 0 ── page 0 ── bbox (114.96, 57.5, 159.42, 68.97) ──
  Font : TT64t00, 11.3pt, bold=False
  Text : Mục lục

─── Block 1 ── page 0 ── bbox (114.96, 78.2, 415.61, 89.97) ──
  Font : TT190t00, 11.3pt, bold=False
  Text : GIỚI THIỆU .............................................................................5

─── Block 2 ── page 0 ── bbox (114.96, 97.43, 415.61, 110.85) ──
  Font : TT190t00, 11.3pt, bold=False
  Text : Chương 1. MỞ ĐẦU ............................................................7

─── Block 3 ── page 0 ── bbox (128.16, 119.96, 415.61, 131.73) ──
  Font : TT190t00, 11.3pt, bold=False
  Text : 1.1. KHÁI NIỆM CƠ BẢN ................................................ 12

─── Block 4 ── page 0 ── bbox (128.16, 140.84, 415.61, 152.61) ──
  Font : TT190t00, 11.3pt, bold=False
  Text : 1.2. ĐỐI TƯỢNG VÀ LỚP................................................ 13

─── Block 5 ── page 0 ── bbox (128.16, 161.6, 415.61, 173.37) ──
  Font : TT190t00, 11.3pt, bold=False
  Text : 1.

## 4. Inspect a Specific Page

Pick a page to see all its blocks in detail (useful for checking heading detection later).

In [7]:
TARGET_PAGE = 4  # ← change this to inspect another page

page_blocks = [b for b in blocks if b.page == TARGET_PAGE]
print(f"Page {TARGET_PAGE}: {len(page_blocks)} block(s)\n")

for i, b in enumerate(page_blocks):
    fi = b.font_info
    font_desc = f"{fi.name}, {fi.size}pt, bold={fi.is_bold}" if fi else "N/A"

    print(f"[{i}] bbox={b.bbox}  font=({font_desc})")
    print(f"    {b.raw_text}")
    print()

Page 4: 6 block(s)

[0] bbox=(93.6, 59.23, 161.3, 74.22)  font=(TT2A6t00, 15.0pt, bold=False)
    Giíi thiÖu

[1] bbox=(93.6, 118.46, 520.62, 190.65)  font=(TT190t00, 11.3pt, bold=False)
    Phần mềm ngày càng lớn và phức tạp và đòi hỏi được cập nhật liên tục để đáp 
ứng những yêu cầu mới của người dùng. Phương pháp lập trình thủ tục truyền 
thống dần trở nên không đáp ứng được những đòi hỏi đó của ngành công nghiệp 
phần mềm. Lập trình hướng đối tượng đã ra đời trong bối cảnh như vậy để hỗ trợ sử 
dụng lại và phát triển các phần mềm qui mô lớn.

[2] bbox=(93.6, 200.06, 520.62, 272.49)  font=(TT190t00, 11.3pt, bold=False)
    Giáo trình này cung cấp cho sinh viên các kiến thức từ cơ bản cho đến một số kỹ 
thuật nâng cao về phương pháp lập trình hướng đối tượng. Giáo trình dùng cho 
sinh viên ngành Công nghệ thông tin đã có kiến thức căn bản về lập trình. Giáo trình 
sử dụng ngôn ngữ lập trình Java để minh họa và đồng thời cũng giới thiệu một số 
kiến thức căn bản của ngôn ngữ này.

[3]